# 📚 Unidad 3: Modelado de Datos - Teoría
## Módulo 01 - Ensemble Methods Avanzados
### Laboratorio - Universidad del Aconcagua

---

## 🎯 Objetivos

1. ✅ Dominar métodos de ensemble (Bagging, Boosting, Stacking)
2. ✅ Implementar Random Forest, XGBoost, LightGBM
3. ✅ Optimizar hiperparámetros
4. ✅ Interpretar modelos complejos

---

## 1️⃣ Introducción a Ensemble Learning

### ¿Qué es Ensemble Learning?

**Ensemble** = Combinar múltiples modelos débiles para crear un modelo fuerte

**Sabiduría de las Multitudes**: Muchos modelos imperfectos juntos superan a un modelo perfecto

### Ventajas

* ✅ Mejor accuracy que modelos individuales
* ✅ Reduce overfitting (variance)
* ✅ Más robusto ante outliers
* ✅ Captura relaciones complejas

---

## 2️⃣ Bagging (Bootstrap Aggregating)

### Concepto

1. Crear N subsamples con replacement (bootstrap)
2. Entrenar un modelo en cada subsample
3. Promediar predicciones (regresión) o votar (clasificación)

### Random Forest

**Mejora sobre Bagging**: Randomiza features en cada split

```python
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,      # Número de árboles
    max_depth=10,          # Profundidad máxima
    max_features='sqrt',   # Features por split
    min_samples_split=5,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
```

**Hiperparámetros clave:**
* `n_estimators`: Más árboles = mejor (hasta plateau)
* `max_depth`: Controla overfitting
* `max_features`: Decorrelaciona árboles
* `min_samples_split`: Mínimo para split

**Feature Importance:**
```python
import pandas as pd

importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(10))
```

---

## 3️⃣ Boosting

### Concepto

1. Entrenar modelo secuencialmente
2. Cada modelo corrige errores del anterior
3. Weighted voting final

### Gradient Boosting

**Idea**: Ajustar residuos iterativamente

### XGBoost (Extreme Gradient Boosting)

```python
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,     # Tasa de aprendizaje
    max_depth=6,
    subsample=0.8,         # Row sampling
    colsample_bytree=0.8,  # Feature sampling
    objective='binary:logistic',
    random_state=42
)

xgb_model.fit(X_train, y_train)
```

**Ventajas XGBoost:**
* ⚡ Muy rápido (paralelización)
* 🎯 Alta precisión
* 🛡️ Maneja missing values
* 🔧 Regularización L1/L2

**Hiperparámetros:**
* `learning_rate`: 0.01-0.3 (más bajo = más árboles)
* `max_depth`: 3-10
* `subsample`: 0.5-1.0
* `colsample_bytree`: 0.5-1.0
* `reg_alpha` (L1), `reg_lambda` (L2)

### LightGBM

**Ventaja**: Aún más rápido que XGBoost

```python
import lightgbm as lgb

lgb_model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,        # Específico de LightGBM
    max_depth=-1,
    random_state=42
)

lgb_model.fit(X_train, y_train)
```

**Cuándo usar:**
* XGBoost: Datasets pequeños-medianos, máxima precisión
* LightGBM: Datasets grandes, velocidad crítica

---

## 4️⃣ Stacking

### Concepto

1. Entrenar múltiples modelos base (level 0)
2. Usar predicciones como features para meta-model (level 1)

```python
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [
    ('rf', RandomForestClassifier(n_estimators=100)),
    ('xgb', xgb.XGBClassifier(n_estimators=100)),
    ('lgb', lgb.LGBMClassifier(n_estimators=100))
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5
)

stacking_model.fit(X_train, y_train)
```

---

## 5️⃣ Optimización de Hiperparámetros

### Grid Search

```python
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}

grid = GridSearchCV(
    xgb.XGBClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train, y_train)
print(f"Best params: {grid.best_params_}")
print(f"Best score: {grid.best_score_:.4f}")
```

### Random Search (más eficiente)

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.29)
}

random_search = RandomizedSearchCV(
    xgb.XGBClassifier(),
    param_dist,
    n_iter=50,
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
```

---

## 6️⃣ Interpretabilidad

### SHAP (SHapley Additive exPlanations)

```python
import shap

# Explicar predicciones
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values, X_test)

# Force plot (predicción individual)
shap.force_plot(explainer.expected_value, shap_values[0], X_test.iloc[0])
```

---

## ✅ Resumen

### Cuándo Usar Cada Método

| Método | Velocidad | Precisión | Interpretabilidad | Uso |
|--------|-----------|-----------|-------------------|-----|
| Random Forest | Media | Alta | Alta | Baseline sólido |
| XGBoost | Alta | Muy Alta | Media | Competiciones |
| LightGBM | Muy Alta | Muy Alta | Media | Big Data |
| Stacking | Baja | Máxima | Baja | Última optimización |

---

**Universidad del Aconcagua 🇦🇷**